In [23]:
import torch, os, cv2, re
import scipy.io as sio
import numpy as np
import pandas as pd

from PIL import Image, ImageDraw, ImageFont

In [24]:
from nltk.stem import WordNetLemmatizer
wnl = WordNetLemmatizer()

def min_max_normalize(arr):
    arr = np.array(arr)
    arr_min = arr.min()
    arr_max = arr.max()
    if arr_max - arr_min == 0:
        return np.zeros_like(arr)
    return (arr - arr_min) / (arr_max - arr_min)

In [25]:
import json, random, csv, time, os, random
import numpy as np
from statistics import mean
from scipy.stats import ttest_rel, ttest_ind
import pandas as pd
from datetime import datetime
from tqdm import tqdm  # for progress bar
import matplotlib.pyplot as plt
from collections import defaultdict
import torchvision.transforms as T

# Load data
with open("../data/coco_caption_concereteness_with_lemmatization.json") as f:
    data = json.load(f)

for i, img in enumerate(data):
    max_score, min_score = 0, 100
    max_id, min_id = 0, 0
    for c, cap in enumerate(img['caption_concreteness']):
        mean_score = mean(cap['concreteness'])
        data[i]['caption_concreteness'][c]['concrete_score'] = mean_score
        data[i]['caption_concreteness'][c]['abstract_score'] = mean_score
        if mean_score > max_score:
            max_score = mean_score
            max_id = cap['id']
        if mean_score < min_score:
            min_score = mean_score
            min_id = cap['id']
    data[i]['concrete_abstact_diff'] = {
        'value' : max_score - min_score,
        'ids': [max_id, min_id],
}

all_attributes = ["matching_score", "word_importances", "char_number", "word_number", "word_frequency", "emap_var", "abstract_score", "concrete_score"]
test_attributes = ["matching_score", "word_importances", "char_number", "word_number", "word_frequency", "emap_var"]

# Filter entries with both concrete and abstract ids
filtered_data = [item for item in data if len(set(item['concrete_abstact_diff']['ids'])) > 1]
filtered_data = sorted(filtered_data, key=lambda x: x['concrete_abstact_diff']['value'], reverse=True)

def extract_attr_lists(data_subset):
    ids = []
    abs_captions = []
    con_captions = []
    value_diffs = []
    abs_vals = {attr: [] for attr in all_attributes}
    con_vals = {attr: [] for attr in all_attributes}

    for entry in data_subset:
        ids.append(entry['image'])
        value_diffs.append(entry['concrete_abstact_diff']['value'])
        con_id, abs_id = entry['concrete_abstact_diff']['ids']
        con_cap = next(c for c in entry['caption_concreteness'] if c['id'] == con_id)
        abs_cap = next(c for c in entry['caption_concreteness'] if c['id'] == abs_id)

        abs_captions.append(abs_cap)
        con_captions.append(con_cap)

        for attr in ["concrete_score", "abstract_score"]:
            if attr in all_attributes:
                abs_vals[attr].append(abs_cap[attr])
                con_vals[attr].append(con_cap[attr])

        if 'char_number' in all_attributes:
            abs_vals['char_number'].append(len(abs_cap['caption']))
            con_vals['char_number'].append(len(con_cap['caption']))

        if 'word_number' in all_attributes:
            abs_vals['word_number'].append(len(abs_cap['caption'].split(' ')))
            con_vals['word_number'].append(len(con_cap['caption'].split(' ')))

        for attr in ["matching_score", "emap_var"]:
            if attr in all_attributes:
                abs_vals[attr].append(abs_cap[attr])
                con_vals[attr].append(con_cap[attr])

        for attr in ["word_importances", "word_frequency"]:
            if attr in all_attributes:
                abs_vals[attr].append(mean(abs_cap[attr]))
                con_vals[attr].append(mean(con_cap[attr]))

    return ids, value_diffs, abs_captions, con_captions, abs_vals, con_vals

ids_all, value_diffs_all, _, _, abs_vals_all, con_vals_all = extract_attr_lists(filtered_data)

experiment = '20250723_193742'

selected_indices = []
saved = pd.read_csv(f"../stimuli/250602-250723_stimuli_average/{experiment}/captions_and_attributes.csv")
saved_ids = saved['image_id'].tolist()
for e, entry in enumerate(filtered_data):
    if entry['image'] in saved_ids:
        selected_indices.append(e)

selected_indices.sort()
selected_entries = [filtered_data[idx] for idx in selected_indices]

ids_selected, value_diffs_selected, abs_captions_selected, con_captions_selected, abs_vals_selected, con_vals_selected = extract_attr_lists(selected_entries)

In [ ]:
analysis_name = "251104_lextale60"
data_all = {}

In [27]:
for idx, (image_id, abs_cap, con_cap) in enumerate(zip(ids_selected, abs_captions_selected, con_captions_selected)):
    # Load image
    img_path = f"../results/{analysis_name}/human/image/images/matched_abs_{image_id.split('/')[-1]}"

    abs_emap = torch.load(f"../data/emaps_resized/{image_id.split('_')[-1].split('.')[0]}_{abs_cap['id']}.pt", map_location="cpu")
    con_emap = torch.load(f"../data/emaps_resized/{image_id.split('_')[-1].split('.')[0]}_{con_cap['id']}.pt", map_location="cpu")

    img = Image.open(img_path).convert("RGB")
    h, w = img.size
    resize = T.Resize((w,h))

    emaps =resize(torch.stack([abs_emap, con_emap], dim=0))

    abs_emap = emaps[0].numpy()
    con_emap = emaps[1].numpy()

    for con, cap, map in zip(['abs', 'con'], [abs_cap, con_cap], [abs_emap, con_emap]):
        human_map = torch.load(f"../results/{analysis_name}/human/image/pts/matched_{con}_{image_id.split('/')[-1].split('.')[0]}_GSmo_41.pt", map_location="cpu").numpy()
        data_all[f"matched_{con}_{image_id.split('/')[-1]}"] = {
            "condition": con,
            "image_id": image_id,
            "caption": cap['caption'],
            "caption_words": [re.sub(r'[^\w\s]', '', tok) for tok in cap['caption'].split()],
            "concreteness": cap["concreteness"],
            "target_words": cap["words"],
            "clip_map": map,
            "human_map": human_map,
            "clip_word_importance_all": cap['word_importances'],
            "clip_word_importance": None,
            "human_word_importance_all": None,
            "human_word_importance": None
        }

In [28]:
import ast 

image_info_segmentation = pd.read_csv("../stimuli/250619-250723_stimuli_experiment/20250723_193742/image_info_segmentation.csv")

for _, row in image_info_segmentation.iterrows():
    trial_id = row['trial_id']

    if "mismatched" in trial_id:
        continue    

    fixations = pd.read_excel(f"../results/{analysis_name}/human/whole_screen/fixations/{trial_id}.xlsx")

    word_spans = ast.literal_eval(row['word_spans'])
    num_fixations = []

    for span in word_spans:
        x1, x2, y1, y2 = span

        # Filter fixations within the bounding box
        y1 = 1024 - y1
        y2 = 1024 - y2
        y1, y2 = y2, y1

        num_fixs = np.sum((fixations['CURRENT_FIX_X'] >= x1) & (fixations['CURRENT_FIX_X'] <= x2) &
                         (fixations['CURRENT_FIX_Y'] >= y1) & (fixations['CURRENT_FIX_Y'] <= y2))
        
        num_fixations.append(num_fixs)

    # Min-max normalization for num_fixations
    data_all[trial_id]['human_word_importance_all'] = (np.array(num_fixations) - np.min(num_fixations)) / (np.max(num_fixations) - np.min(num_fixations))

    # Only analyze words with concreteness ratings
    target_words = data_all[trial_id]['target_words']
    word_lists = data_all[trial_id]['caption_words']

    assert len(data_all[trial_id]['human_word_importance_all']) == len(word_lists) and\
        len(data_all[trial_id]['clip_word_importance_all']) == len(word_lists), f"Length mismatch in {trial_id}"

    clip_word_importance = []
    human_word_importance = []

    for word, clip_importance, human_importance in zip(word_lists, data_all[trial_id]['clip_word_importance_all'], data_all[trial_id]['human_word_importance_all']):
        if word in target_words or wnl.lemmatize(word) in target_words:
            clip_word_importance.append(clip_importance)
            human_word_importance.append(human_importance)

    data_all[trial_id]['clip_word_importance'] = clip_word_importance
    data_all[trial_id]['human_word_importance'] = human_word_importance

In [29]:
# import ast 

# image_info_segmentation = pd.read_csv("../stimuli/250619-250723_stimuli_experiment/20250723_193742/image_info_segmentation.csv")

# for _, row in image_info_segmentation.iterrows():
#     trial_id = row['trial_id']

#     if "mismatched" in trial_id:
#         continue    

#     fixations = pd.read_excel(f"../results/{analysis_name}/human/whole_screen/fixations/{trial_id}.xlsx")

#     word_spans = ast.literal_eval(row['word_spans'])

#     # Only analyze words with concreteness ratings
#     target_words = data_all[trial_id]['target_words']
#     # word_lists = data_all[trial_id]['caption_words']
#     word_lists = ast.literal_eval(row['word_list'])
    
#     num_fixations = []
#     num_fixations_all = []

#     clip_word_importance = []

#     for word, span, clip_importance in zip(word_lists, word_spans, data_all[trial_id]['clip_word_importance_all']):

#         x1, x2, y1, y2 = span

#         # Filter fixations within the bounding box
#         y1 = 1024 - y1
#         y2 = 1024 - y2
#         y1, y2 = y2, y1

#         num_fixs = np.sum((fixations['CURRENT_FIX_X'] >= x1) & (fixations['CURRENT_FIX_X'] <= x2) &
#                          (fixations['CURRENT_FIX_Y'] >= y1) & (fixations['CURRENT_FIX_Y'] <= y2))

#         num_fixations_all.append(num_fixs)
        
#         if word in target_words or wnl.lemmatize(word) in target_words:
#             clip_word_importance.append(clip_importance)
#             num_fixations.append(num_fixs)
        
#     # Min-max normalization for num_fixations
#     assert len(data_all[trial_id]['human_word_importance_all']) == len(word_lists) and\
#     len(data_all[trial_id]['clip_word_importance_all']) == len(word_lists), f"Length mismatch in {trial_id}"

#     data_all[trial_id]['clip_word_importance'] = min_max_normalize(clip_word_importance)
#     data_all[trial_id]['human_word_importance'] = min_max_normalize(num_fixations)

In [30]:
for trial_id, info in data_all.items():
    data_all[trial_id]['map_pcc'] = np.corrcoef(info['clip_map'].flatten(), info['human_map'].flatten())[0, 1]
    # only words with concreteness ratings
    data_all[trial_id]['word_importance_pcc'] = np.corrcoef(info['clip_word_importance'], info['human_word_importance'])[0, 1]
    data_all[trial_id]['concatenate_pcc'] = np.corrcoef(
        np.concatenate([info['clip_map'].flatten(), info['clip_word_importance']]),
        np.concatenate([info['human_map'].flatten(), info['human_word_importance']])
    )[0, 1]
    # all words
    data_all[trial_id]['word_importance_pcc_all'] = np.corrcoef(info['clip_word_importance_all'], info['human_word_importance_all'])[0, 1]
    data_all[trial_id]['concatenate_pcc_all'] = np.corrcoef(
        np.concatenate([info['clip_map'].flatten(), info['clip_word_importance_all']]),
        np.concatenate([info['human_map'].flatten(), info['human_word_importance_all']])
    )[0, 1]

In [31]:
# T-test analysis
for measurement in ['map_pcc', 'word_importance_pcc', 'concatenate_pcc', 'word_importance_pcc_all', 'concatenate_pcc_all']:
    con_pcc_scores = [info[measurement] for trial_id, info in data_all.items() if info['condition'] == 'con']
    abs_pcc_scores = [info[measurement] for trial_id, info in data_all.items() if info['condition'] == 'abs']

    # Perform t-test
    t_stat, p_value = ttest_ind(con_pcc_scores, abs_pcc_scores, equal_var=False)

    print(f"({measurement}) T-statistic: {t_stat}, P-value: {p_value}")

(map_pcc) T-statistic: -0.21799858338691072, P-value: 0.8277120843881597
(word_importance_pcc) T-statistic: -0.23827243093213346, P-value: 0.8119789665755481
(concatenate_pcc) T-statistic: -0.2176902938578019, P-value: 0.827951879918134
(word_importance_pcc_all) T-statistic: 1.8479882219809063, P-value: 0.06651977647776437
(concatenate_pcc_all) T-statistic: -0.21761115379565146, P-value: 0.8280134404184358


### Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import font_manager

def draw_highlighted_caption(caption, 
                             target_words,
                             concreteness_all, importance_all, 
                             start_pos=(60, 24*3), fontsize=14,
                             coloring=True,
                             base_color=(0, 0, 1),  # matplotlib uses 0–1 floats
                             highlight_color=(223/255, 245/255, 39/255),
                             save_path="highlighted_caption.png"):
    """
    Matplotlib version of highlighted text renderer.
    Creates its own figure/axes and saves result to file.
    """

    # Create fig/ax
    fig, ax = plt.subplots(figsize=(12.8, 1.92), dpi=129.94923858)
    ax.set_xlim(0, 1280)
    ax.set_ylim(0, 192)
    ax.axis('off')

    x, y = start_pos
    words = caption.strip('.,!?\'" ').split()

    renderer = fig.canvas.get_renderer()
    inv = ax.transData.inverted()

    for word, importance in zip(words, importance_all):
        word_clean = word.strip('.,!?\'"')
        vocab = word_clean.lower()

        # default black text
        color = (0, 0, 0, 1)

        if coloring:
            # background highlight rectangle
            highlight_alpha = importance
            bg_color = highlight_color + (highlight_alpha,)

            text = ax.text(x, y, word, fontsize=fontsize, color='black', va='top')
            bbox = text.get_window_extent(renderer=renderer)
            text.remove()

            bbox_data = bbox.transformed(inv)
            rect_width, rect_height = bbox_data.width, bbox_data.height

            rect = patches.Rectangle((x-0.05, y-rect_height*0.2),
                                     rect_width+0.1, rect_height*1.2,
                                     color=bg_color, zorder=1)
            ax.add_patch(rect)

            if vocab in target_words:
                idx = target_words.index(vocab)
                concreteness = concreteness_all[idx] / 5
                text_alpha = concreteness
                color = base_color + (text_alpha,)

        # draw text
        ax.text(x, y, word, fontsize=fontsize,
                color=color, zorder=2)

        # move x forward
        tmp = ax.text(x, y, word + " ", fontsize=fontsize, alpha=0)
        bbox = tmp.get_window_extent(renderer=renderer)
        tmp.remove()
        bbox_data = bbox.transformed(inv)
        word_width = bbox_data.width
        x += word_width

    # Save and close
    plt.savefig(save_path, bbox_inches="tight", pad_inches=0)
    plt.close(fig)

    return save_path

output_dir = f'../results/{analysis_name}/human/word_importance'
os.makedirs(output_dir, exist_ok=True)

for trial_id, info in data_all.items():
    # Abstract caption
    draw_highlighted_caption(info['caption'],
                            info['target_words'],
                            info['concreteness'],
                            info['human_word_importance'],
                         save_path=os.path.join(output_dir, trial_id+".png"))